In [1]:
import asyncio
import csv
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import time
import nest_asyncio
from pathlib import Path
import ast

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Import your existing conversation generator
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')
from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator

class MovieConversationGenerator:
    """Generator for movie recommendation conversations based on CSV data"""
    
    def __init__(self, base_config: ConversationConfig, user_prompt_template_path: str, terminal_signal: str = "[[TERMINATE CHAT]]"):
        self.base_config = base_config
        self.terminal_signal = terminal_signal
        
        # Load the user meta prompt template
        with open(user_prompt_template_path, 'r') as f:
            self.user_prompt_template = f.read()
    
    def load_csv_data(self, csv_path: str) -> pd.DataFrame:
        """Load and parse the CSV data"""
        df = pd.read_csv(csv_path)
        
        # Parse the conversation column (assuming it's stored as string representation of list)
        def parse_conversation(conv_str):
            try:
                # Handle the conversation string - it might be a JSON string or Python literal
                if isinstance(conv_str, str):
                    return ast.literal_eval(conv_str)
                return conv_str
            except (ValueError, SyntaxError) as e:
                logger.warning(f"Failed to parse conversation: {e}")
                return []
        
        df['conversation_parsed'] = df['conversation'].apply(parse_conversation)
        return df
    
    def create_custom_config(self, conversation: List[Dict], ground_truth: str) -> ConversationConfig:
        """Create a custom config with conversation-specific user meta prompt"""
        # Convert conversation to a readable format
        conv_text = ""
        for msg in conversation:
            role = msg['role']
            content = msg['content']
            if role == 'user':
                conv_text += f"User: {content}\n"
            elif role == 'assistant':
                conv_text += f"Assistant: {content}\n"
        
        # Fill in the template with the conversation data
        custom_user_prompt = self.user_prompt_template.format(
            conversation=conv_text.strip(),
            ground_truth=ground_truth,
            terminal_signal=self.terminal_signal,
            chat_history="{chat_history}"  # Keep this placeholder for UserSimulator
        )
        
        # Create a new config with the custom prompt
        custom_config = ConversationConfig(
            assistant_meta_prompt=self.base_config.assistant_meta_prompt,
            user_meta_prompt=custom_user_prompt,
            max_total_turns=self.base_config.max_total_turns,
            max_gen_workers=self.base_config.max_gen_workers,
            local_model_path=self.base_config.local_model_path,
            base_model_path=self.base_config.base_model_path,
            assistant_generation_kwargs=self.base_config.assistant_generation_kwargs,
            user_generation_kwargs=self.base_config.user_generation_kwargs,
            batch_size=self.base_config.batch_size,
            enable_batching=self.base_config.enable_batching
        )
        
        return custom_config
    
    async def generate_single_movie_conversation(self, dialog_id: str, conversation: List[Dict], ground_truth: str) -> Optional[Dict]:
        """Generate a single conversation based on the movie data"""
        try:
            logger.info(f"Generating conversation for dialog_id: {dialog_id}")
            
            # Create custom config for this specific conversation
            custom_config = self.create_custom_config(conversation, ground_truth)
            
            # Create a new generator with the custom config
            generator = MultiTurnConversationGenerator(custom_config)
            
            # Use a simple initial prompt since all context is in user_meta_prompt
            initial_prompt = "I'm looking for a movie recommendation."
            
            # Generate the conversation
            generated_conv = await generator.generate_single_conversation(initial_prompt)
            
            if generated_conv:
                result = {
                    'dialog_id': dialog_id,
                    'ground_truth': ground_truth,
                    'original_conversation': conversation,
                    'generated_conversation': generated_conv,
                    'status': 'success'
                }
                logger.info(f"Successfully generated conversation for {dialog_id}")
                return result
            else:
                logger.warning(f"Failed to generate conversation for {dialog_id}")
                return {
                    'dialog_id': dialog_id,
                    'ground_truth': ground_truth,
                    'original_conversation': conversation,
                    'generated_conversation': None,
                    'status': 'failed'
                }
                
        except Exception as e:
            logger.error(f"Error generating conversation for {dialog_id}: {str(e)}")
            return {
                'dialog_id': dialog_id,
                'ground_truth': ground_truth,
                'original_conversation': conversation,
                'generated_conversation': None,
                'status': 'error',
                'error': str(e)
            }
    
    async def generate_conversations_batch(self, df: pd.DataFrame, batch_size: Optional[int] = None) -> List[Dict]:
        """Generate conversations for all rows in the dataframe"""
        if batch_size is None:
            batch_size = self.base_config.batch_size
        
        total_rows = len(df)
        logger.info(f"Starting batch generation for {total_rows} conversations")
        
        # Process in smaller batches to manage memory
        results = []
        
        for i in range(0, total_rows, batch_size):
            batch_end = min(i + batch_size, total_rows)
            batch_df = df.iloc[i:batch_end]
            
            logger.info(f"Processing batch {i//batch_size + 1}: rows {i+1}-{batch_end}")
            
            # Create tasks for this batch
            tasks = []
            for idx, row in batch_df.iterrows():
                task = self.generate_single_movie_conversation(
                    dialog_id=row['dialog_id'],
                    conversation=row['conversation_parsed'],
                    ground_truth=row['ground_truth']
                )
                tasks.append(task)
            
            # Execute batch
            start_time = time.time()
            batch_results = await asyncio.gather(*tasks, return_exceptions=True)
            end_time = time.time()
            
            # Handle any exceptions in the batch
            for j, result in enumerate(batch_results):
                if isinstance(result, Exception):
                    logger.error(f"Exception in batch {i//batch_size + 1}, item {j}: {result}")
                    # Create error result
                    row_idx = i + j
                    row = df.iloc[row_idx]
                    error_result = {
                        'dialog_id': row['dialog_id'],
                        'ground_truth': row['ground_truth'],
                        'original_conversation': row['conversation_parsed'],
                        'generated_conversation': None,
                        'status': 'exception',
                        'error': str(result)
                    }
                    results.append(error_result)
                else:
                    results.append(result)
            
            logger.info(f"Batch {i//batch_size + 1} completed in {end_time - start_time:.2f} seconds")
        
        successful = sum(1 for r in results if r['status'] == 'success')
        logger.info(f"Batch generation complete: {successful}/{total_rows} successful")
        
        return results
    
    def save_results_to_csv(self, results: List[Dict], output_path: str):
        """Save results to CSV file"""
        # Prepare data for CSV
        csv_data = []
        for result in results:
            # Convert generated conversation to string for CSV storage
            generated_conv_str = json.dumps(result['generated_conversation']) if result['generated_conversation'] else None
            original_conv_str = json.dumps(result['original_conversation'])
            
            csv_row = {
                'dialog_id': result['dialog_id'],
                'ground_truth': result['ground_truth'],
                'original_conversation': original_conv_str,
                'generated_conversation': generated_conv_str,
                'status': result['status']
            }
            
            # Add error information if present
            if 'error' in result:
                csv_row['error'] = result['error']
            
            csv_data.append(csv_row)
        
        # Write to CSV
        df_output = pd.DataFrame(csv_data)
        df_output.to_csv(output_path, index=False)
        logger.info(f"Results saved to {output_path}")
        
        # Print summary
        status_counts = df_output['status'].value_counts()
        logger.info(f"Summary: {status_counts.to_dict()}")

# Configuration for movie conversation generation
def create_movie_config():
    """Create base configuration for movie recommendation conversations"""
    return ConversationConfig(
        assistant_meta_prompt="You are a helpful movie recommendation assistant. Provide personalized movie suggestions based on user preferences and engage in natural conversation about movies.",
        # This will be overridden for each conversation with custom context
        user_meta_prompt="You are a user looking for movie recommendations. Respond naturally based on the conversation context.",
        max_total_turns=12,
        max_gen_workers=4,
        local_model_path="/home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test",
        base_model_path="meta-llama/Llama-3.2-1B-Instruct",
        assistant_generation_kwargs={
            "temperature": 0.7,
            "max_tokens": 512
        },
        user_generation_kwargs={
            "model": "us.anthropic.claude-sonnet-4-20250514-v1:0",
            "temperature": 0.8,
            "max_tokens": 256,
            "num_retries": 5
        },
        batch_size=50,  # Smaller batch size for stability
        enable_batching=True
    )




INFO 07-10 05:35:07 [__init__.py:244] Automatically detected platform cuda.


In [2]:
async def test_single_conversation():
    """Test with a single conversation"""
    config = create_movie_config()
    generator = MovieConversationGenerator(
        base_config=config,
        user_prompt_template_path="../prompts/test_user_prompt.txt",
        terminal_signal="[[TERMINATE CHAT]]"
    )
    
    # Test data (based on your example)
    test_conversation = [
        {'role': 'assistant', 'content': "Hi! I'm here to help you chose a movie!"},
        {'role': 'user', 'content': 'Terrific'},
        {'role': 'assistant', 'content': 'What are some genres you like? What was the last movie you saw?'},
        {'role': 'user', 'content': 'the last movie i saw in the theater was "Hustlers". I generally like comedy, drama and documentaries'},
    ]
    
    result = await generator.generate_single_movie_conversation(
        dialog_id="test_001",
        conversation=test_conversation,
        ground_truth="A Beautiful Day in the Neighborhood"
    )
    
    print("Test result:", result['status'])
    if result['generated_conversation']:
        print(f"Generated {len(result['generated_conversation'])} messages")
        for msg in result['generated_conversation']:
            print(f"{msg['role']}: {msg['content'][:100]}...")
            
    return result

# Run tests
# print("🧪 Testing single conversation...")
# await test_single_conversation()

INFO: Generating conversation for dialog_id: test_001


🧪 Testing single conversation...
🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-10 05:24:53 [config.py:823] This model supports multiple tasks: {'classify', 'score', 'reward', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 07-10 05:24:53 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-10 05:24:53 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 07-10 05:24:55 [core.py:455] Waiting for init message from front-end.
INFO 07-10 05:24:55 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=1 pid=18969) INFO 07-10 05:25:00 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=4 pid=18972) INFO 07-10 05:25:00 [cuda.py:252] Using Flash Attention backend on V1 engine.
(VllmWorker rank=7 pid=18975) INFO 07-10 05:25:00 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=5 pid=18973) INFO 07-10 05:25:00 [cuda.py:252] Using Flash Attention backend on V1 engine.
(VllmWorker rank=2 pid=18970) (VllmWorker rank=1 pid=18969) INFO 07-10 05:25:00 [weight_utils.py:292] Using model weights format ['*.safetensors']
INFO 07-10 05:25:00 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=0 pid=18968) INFO 07-10 05:25:00 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=0 pid=18968) INFO 07-10 05:25:00 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=7 pid=18975) INFO 07-10 05:25:00 [default_loader.py:272] Loading weights took 0.10 se

INFO: ✅ Successfully initialized vLLM with LoRA: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test


...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Turn 6: Generating user response...


INFO: Successfully generated conversation for test_001


✅ Conversation completed with 13 messages
Test result: success
Generated 13 messages
user: I'm looking for a movie recommendation....
assistant: I'd be happy to help. What kind of movies do you usually enjoy watching? (e.g. action, comedy, drama...
user: I like comedy, drama and documentaries. The last thing I saw in theaters was Hustlers...
assistant: I'm a big fan of documentaries myself. What genre do you usually stick to? (e.g. biographical, docum...
user: I don't really stick to one type - I'm pretty open. Could be about true crime, music, social issues,...
assistant: Okay, no problem! I have a few suggestions for you. Have you seen "The Pursuit of Happyness" or "The...
user: I've seen The Social Network but not The Pursuit of Happyness. What's that one about?...
assistant: 12 years a slave is a great movie. It's based on a true story about a slave who escapes from slavery...
user: I think you might be mixing up movies - 12 Years a Slave is definitely not a documentary and it's ab

{'dialog_id': 'test_001',
 'ground_truth': 'A Beautiful Day in the Neighborhood',
 'original_conversation': [{'role': 'assistant',
   'content': "Hi! I'm here to help you chose a movie!"},
  {'role': 'user', 'content': 'Terrific'},
  {'role': 'assistant',
   'content': 'What are some genres you like? What was the last movie you saw?'},
  {'role': 'user',
   'content': 'the last movie i saw in the theater was "Hustlers". I generally like comedy, drama and documentaries'}],
 'generated_conversation': [{'role': 'user',
   'content': "I'm looking for a movie recommendation."},
  {'role': 'assistant',
   'content': "I'd be happy to help. What kind of movies do you usually enjoy watching? (e.g. action, comedy, drama, romance, etc.)?"},
  {'role': 'user',
   'content': 'I like comedy, drama and documentaries. The last thing I saw in theaters was Hustlers'},
  {'role': 'assistant',
   'content': "I'm a big fan of documentaries myself. What genre do you usually stick to? (e.g. biographical, doc

In [2]:
async def main():
    """Main function to run the movie conversation generation"""
    
    # File paths
    csv_input_path = "../datasets/inspired/multiturn_form/test.csv"
    user_prompt_template_path = "../prompts/test_user_prompt.txt"
    csv_output_path = "multiturn_test/llama3_2_1B/inspired/generated_movie_conversations.csv"
    
    # Create base configuration
    config = create_movie_config()
    
    # Initialize generator with template path
    movie_generator = MovieConversationGenerator(
        base_config=config,
        user_prompt_template_path=user_prompt_template_path,
        terminal_signal="TERMINATE"
    )
    
    # Load data
    logger.info("Loading CSV data...")
    df = movie_generator.load_csv_data(csv_input_path)
    logger.info(f"Loaded {len(df)} conversations from CSV")
    
    # Generate conversations
    logger.info("Starting conversation generation...")
    start_time = time.time()
    
    results = await movie_generator.generate_conversations_batch(df)
    
    end_time = time.time()
    logger.info(f"Total generation time: {end_time - start_time:.2f} seconds")
    
    # Save results
    logger.info("Saving results...")
    movie_generator.save_results_to_csv(results, csv_output_path)
    
    logger.info("Process complete!")
    
    return results

# Run main process
print("🚀 Starting main generation process...")
results = await main()

INFO: Loading CSV data...
INFO: Loaded 99 conversations from CSV
INFO: Starting conversation generation...
INFO: Starting batch generation for 99 conversations
INFO: Processing batch 1: rows 1-50
INFO: Generating conversation for dialog_id: 20191127-224739_530_live.pkl


🚀 Starting main generation process...
🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-10 05:35:26 [config.py:823] This model supports multiple tasks: {'generate', 'reward', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 07-10 05:35:26 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-10 05:35:26 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 07-10 05:35:28 [core.py:455] Waiting for init message from front-end.
INFO 07-10 05:35:28 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=2 pid=33544) INFO 07-10 05:35:33 [default_loader.py:272] Loading weights took 0.11 seconds
(VllmWorker rank=2 pid=33544) INFO 07-10 05:35:33 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=3 pid=33545) INFO 07-10 05:35:33 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=0 pid=33542) INFO 07-10 05:35:34 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=5 pid=33547) INFO 07-10 05:35:34 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=0 pid=33542) INFO 07-10 05:35:34 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=3 pid=33545) INFO 07-10 05:35:34 [default_loader.py:272] Loading weights took 0.10 seconds
(VllmWorker rank=6 pid=33548) INFO 07-10 05:35:34 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=3 pid=33545) INFO 07-10 05:35:34 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=5 

INFO: ✅ Successfully initialized vLLM with LoRA: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO: Generating conversation for dialog_id: 20191130-081727_440_live.pkl


...

(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492] WorkerProc failed to start.
(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492] Traceback (most recent call last):
(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492]   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/executor/multiproc_executor.py", line 466, in worker_main
(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492]     worker = WorkerProc(*args, **kwargs)
(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492]   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/executor/multiproc_executor.py", line 362, in __init__
(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492]     self.worker.init_device()
(VllmWorker rank=2 pid=39640) ERROR 07-10 05:37:05 [multiproc_executor.py:492]   File "/home/sagemaker-

Process EngineCore_0:
Traceback (most recent call last):
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 519, in run_engine_core
    raise e
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 506, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 390, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 76, in __init__
    sel

🤖 Initializing conversation generator...
🚀 Initializing LoRA assistant with 8 GPU(s)...
   Base model: meta-llama/Llama-3.2-1B-Instruct
   LoRA model: /home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test
INFO 07-10 05:37:06 [config.py:823] This model supports multiple tasks: {'generate', 'reward', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 07-10 05:37:06 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-10 05:37:06 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 07-10 05:37:06 [core.py:455] Waiting for init message from front-end.
INFO 07-10 05:37:06 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 07-10 05:37:07 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f0294123370>
(VllmWorker rank=0 pid=40047) INFO 07-10 05:37:07 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_b93d9687'), local_subscribe_addr='ipc:///tmp/c71f0eba-0383-48d7-88a4-5da68ea77d3b', remote_subscribe_addr=None, remote_addr_ipv6=False)
WARNING 07-10 05:37:07 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f0294123a30>
(VllmWorker rank=1 pid=40051) INFO 07-10 05:37:07 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_ad05daac'), local_subscribe_addr='ipc:///tmp/8b97546e-c173-4aae-a46e-9